# Phase 1 — Architecture 02: Hybrid Local/Global Attention + GQA (decoder-only)

**Idea (one line):** most layers only look at the last 256 tokens (sliding window), every 4th layer sees the whole
context, and K/V heads are shared across groups of query heads (GQA, 8 query heads : 2 KV heads).

| | Model 01 | **Model 02 (this notebook)** |
|---|---|---|
| layers / d_model / query heads | 12 / 512 / 8 | same |
| KV heads | 8 (MHA) | **2 (GQA)** |
| attention per layer | all full | **layers 3, 7, 11 (0-indexed) full, other 9 use a 256-token sliding window** |
| d_ff (SwiGLU) | 1408 | **1664** (spends exactly the params GQA freed → same total ≈ 54.9M) |
| positional encoding | *none in practice* (RoPE was implemented but never wired in) | **RoPE, actually applied** |

**Why this design (for the report):** legal text is mostly formulaic local phrasing plus a few long-range lookups
(party names, section numbers, earlier citations). Local layers handle the first, the few global layers the second.
GQA and local attention cut KV cost along different axes (heads vs. sequence positions).
Topics from the lectures: sliding-window / local / sparse attention, MHA vs MQA vs GQA, KV-cache size vs quality, RoPE.

**What is fixed compared with the notebook for model 01** (details in the final message):
1. fp16 autocast really turned on (01 trained in fp32 by accident) 2. weight decay really applied
3. the batch size now comes from `TrainConfig` (01 silently used 8 sequences per micro-batch) 4. RoPE is wired in
5. exact per-source validation loss + a final evaluation 6. generation: `<eos>` banned (always 128 words),
n-gram blocking against loops, vectorised repetition penalty, ID/word-count checks.

**Run order:** top to bottom. Cell 18 (run settings) is the only place you may want to edit.

In [1]:
# Cell 0 — only needed if the shards must be re-encoded from Hugging Face
!pip install -q ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 3.7 MB/s eta 0:00:00


In [2]:
# Cell 1 — Hugging Face login (only used when re-encoding the shards)
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient



try:
    # _tok = UserSecretsClient().get_secret("HF")  
    # secret_label = "HF"
    user_secrets = UserSecretsClient()
    # secret_value = UserSecretsClient().get_secret(secret_label)
    secret_value_0 = user_secrets.get_secret("HF")
    os.environ["HF_TOKEN"] = secret_value_0
    login(secret_value_0)
    print("HF login ok")
except Exception as e:
    print("HF login skipped:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [3]:
# Cell 2 — Imports, paths, device
import os, sys, json, math, time, random, hashlib, shutil, zipfile
from pathlib import Path
from dataclasses import dataclass, asdict, field, replace
from typing import Optional, Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")
from kaggle_secrets import UserSecretsClient

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# fp16 autocast on ANY cuda GPU (a T4 has fp16 tensor cores).
# Model 01 used `not torch.cuda.is_bf16_supported()`, which is False on a T4 -> it silently trained in fp32.
USE_FP16 = device.type == "cuda"
print("Device:", device, "| fp16 autocast:", USE_FP16)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("WORK_DIR:", WORK_DIR, "| TOKENIZER_DIR:", TOKENIZER_DIR)

Device: cuda | fp16 autocast: True
GPU: Tesla T4
WORK_DIR: /kaggle/working | TOKENIZER_DIR: /kaggle/input/datasets/ascyrax/capstone-tokenizer-large


In [4]:
# Cell 3 — Load YOUR tokenizer (unchanged from model 01)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID, "UNK", UNK_ID)

_s = "The Supreme Court of India held that the appeal must be allowed."
print("round-trip ok:", tokenizer.decode(tokenizer.encode(_s, allow_special=False)) == _s)

vocab_size: 32000 | BOS 2 EOS 3 PAD 0 UNK 1
round-trip ok: True


In [5]:
# Cell 4 — Model config (serialisable; saved inside every checkpoint)
@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 12
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                 # < n_head  => GQA (each KV head is shared by n_head//n_kv_head query heads)
    head_dim: int = 64
    ff_mult: float = 8 / 3             # SwiGLU hidden size ~ (8/3)*d_model, rounded up to a multiple of 64
    d_ff: Optional[int] = None         # if set, overrides ff_mult
    max_seq_len: int = 1024

    # --- attention pattern ---
    local_window: int = 0              # 0 => every layer is full causal attention
    global_layers: List[int] = field(default_factory=list)   # 0-indexed layers that stay FULL when local_window > 0

    # --- components ---
    norm_type: str = "rmsnorm"         # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"             # "rope" | "none"
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"           # "swiglu" | "gelu"
    mixer_type: str = "attention"      # token-mixer registry key (swap in other mixers later)
    channel_mixer_type: str = "mlp"    # channel-mixer registry key

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("rope", "none")
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)
        if self.local_window > 0:
            assert all(0 <= i < self.n_layer for i in self.global_layers)

    def is_global(self, layer_idx: int) -> bool:
        return self.local_window <= 0 or layer_idx in self.global_layers

    def window_for(self, layer_idx: int) -> Optional[int]:
        return None if self.is_global(layer_idx) else self.local_window


def preset_baseline_rope(vocab_size=None, max_seq_len=1024):
    """Model 01's shape (MHA 8/8, d_ff 1408, all layers full) but with RoPE really applied.
    Use this to train a clean controlled baseline for the ablation if you have GPU time."""
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, max_seq_len=max_seq_len)


def preset_hybrid_gqa(vocab_size=None, max_seq_len=1024):
    """Architecture 02: GQA (2 KV heads) + sliding window 256 on 9 of 12 layers + wider FFN to keep params equal."""
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, max_seq_len=max_seq_len,
                       n_kv_head=2, d_ff=1664, local_window=256, global_layers=[3, 7, 11])

print(json.dumps(asdict(preset_hybrid_gqa()), indent=1))

{
 "vocab_size": 32000,
 "n_layer": 12,
 "d_model": 512,
 "n_head": 8,
 "n_kv_head": 2,
 "head_dim": 64,
 "ff_mult": 2.6666666666666665,
 "d_ff": 1664,
 "max_seq_len": 1024,
 "local_window": 256,
 "global_layers": [
  3,
  7,
  11
 ],
 "norm_type": "rmsnorm",
 "norm_eps": 1e-05,
 "pos_type": "rope",
 "rope_theta": 10000.0,
 "mlp_type": "swiglu",
 "mixer_type": "attention",
 "channel_mixer_type": "mlp",
 "bias": false,
 "attn_dropout": 0.0,
 "resid_dropout": 0.0,
 "embd_dropout": 0.0,
 "tie_embeddings": true,
 "init_std": 0.02,
 "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers
class LayerNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: scale by 1/rms(x) only (no mean centering, no bias). Computed in fp32 for stability."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)
        return (x / rms) * self.g


def build_norm(cfg: ModelConfig):
    if cfg.norm_type == "layernorm": return LayerNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":   return RMSNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Rotary position embeddings (RoPE)
class RoPE(nn.Module):
    """Rotates each (even, odd) pair of a q/k vector by an angle proportional to its position.
    Then q_i . k_j depends only on the RELATIVE offset i-j. Tables are precomputed; math runs in fp32."""
    def __init__(self, head_dim: int, max_seq_len: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)       # [max_seq_len, head_dim/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):                      # x: [B,H,T,D]; cos,sin: [*,1,T,D/2]
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids):
        """q: [B,Hq,T,D], k: [B,Hkv,T,D], position_ids: [B,T] (absolute positions of these T tokens)."""
        cos = self.cos[position_ids].unsqueeze(1).float()   # [B,1,T,D/2]
        sin = self.sin[position_ids].unsqueeze(1).float()
        dt = q.dtype
        return (self._rotate(q.float(), cos, sin).to(dt),
                self._rotate(k.float(), cos, sin).to(dt))

In [8]:
# Cell 7 — Attention: GQA + optional sliding window + KV cache
_MASK_CACHE = {}

def causal_window_mask(T: int, S: int, window: Optional[int], dev) -> torch.Tensor:
    """Bool mask [1,1,T,S], True = query may attend to key.
    The T queries are the LAST T positions of the S keys (so it also works with a KV cache).
    Query at absolute position p attends keys j with  j <= p  and  (window is None or p - j < window),
    i.e. itself + the previous (window-1) tokens."""
    key = (T, S, window, str(dev))
    m = _MASK_CACHE.get(key)
    if m is None:
        qpos = torch.arange(S - T, S, device=dev).view(T, 1)
        kpos = torch.arange(S, device=dev).view(1, S)
        allowed = kpos <= qpos
        if window is not None:
            allowed = allowed & ((qpos - kpos) < window)
        m = allowed.view(1, 1, T, S)
        if T > 1:                       # cache the big square-ish masks only
            _MASK_CACHE[key] = m
    return m


class Attention(nn.Module):
    """Causal multi-head self-attention.
    - GQA: n_kv_head K/V heads, each shared by n_head // n_kv_head query heads (KV cache shrinks by that factor).
    - Local layers: each token sees only the last `window` tokens. Global layers: window=None (full causal).
    - KV cache stores the (already RoPE-rotated) K and V."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.cfg = cfg
        self.layer_idx = layer_idx
        self.window = cfg.window_for(layer_idx)
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.attn_dropout)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)            # [B,n,T,D]

    def forward(self, x, rope=None, position_ids=None, attention_mask=None, kv_cache=None):
        """x: [B,T,d]; position_ids: [B,T]; attention_mask: [B,S] (1 keep / 0 pad, S = cache + T);
        kv_cache: (k_prev, v_prev) each [B,n_kv_head,S_prev,D]. Returns (out, new_cache)."""
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)

        if rope is not None:                                              # RoPE on the NEW q/k only
            q, k = rope(q, k, position_ids)

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        S = k.shape[2]

        if self.n_kv_head != self.n_head:                                 # GQA: share each K/V head
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)

        window_binds = self.window is not None and S > self.window
        need_mask = (attention_mask is not None) or window_binds or (T > 1 and S != T)

        allowed = None
        if need_mask:
            allowed = causal_window_mask(T, S, self.window, x.device)     # [1,1,T,S]
            if attention_mask is not None:
                allowed = allowed & attention_mask[:, None, None, :].bool()
                # a left-padding query would see nothing (-> NaN). Let every query see itself (harmless for real tokens).
                qpos = torch.arange(S - T, S, device=x.device).view(T, 1)
                kpos = torch.arange(S, device=x.device).view(1, S)
                allowed = allowed | (kpos == qpos).view(1, 1, T, S)

        p = self.attn_dropout.p if self.training else 0.0
        if self.cfg.use_sdpa:
            if need_mask:
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=p, is_causal=False)
            else:                                                         # plain causal (global layer, training)
                out = F.scaled_dot_product_attention(q, k, v, dropout_p=p, is_causal=(T > 1))
        else:                                                             # manual reference path (used in tests)
            if allowed is None:
                allowed = causal_window_mask(T, S, None, x.device)
            scores = (q @ k.transpose(-2, -1)) * self.scale
            scores = scores.masked_fill(~allowed, float("-inf"))
            probs = self.attn_dropout(F.softmax(scores.float(), dim=-1).to(q.dtype))
            out = probs @ v

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Feed-forward blocks
class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )"""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

In [10]:
# Cell 9 — Registries (plug in new token mixers / channel mixers later) + transformer block
TOKEN_MIXERS, CHANNEL_MIXERS = {}, {}

def register_token_mixer(name):
    def deco(fn): TOKEN_MIXERS[name] = fn; return fn
    return deco

def register_channel_mixer(name):
    def deco(fn): CHANNEL_MIXERS[name] = fn; return fn
    return deco

@register_token_mixer("attention")
def _build_attention(cfg, layer_idx):
    return Attention(cfg, layer_idx)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    if cfg.mlp_type == "swiglu": return SwiGLU(cfg)
    if cfg.mlp_type == "gelu":   return GELUMLP(cfg)
    raise ValueError(cfg.mlp_type)


class Block(nn.Module):
    """Pre-norm block:  x = x + mixer(norm(x));  x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.norm1, self.norm2 = build_norm(cfg), build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg, layer_idx)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)
    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [11]:
# Cell 10 — Backbone, LM head, weight init
class Backbone(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def forward(self, input_ids, attention_mask=None, position_ids=None, kv_caches=None):
        B, T = input_ids.shape
        if position_ids is None:                       # default: continue after whatever is in the cache
            past = kv_caches[0][0].shape[2] if kv_caches is not None else 0
            position_ids = (past + torch.arange(T, device=input_ids.device)).unsqueeze(0).expand(B, T)
        x = self.emb_drop(self.tok_emb(input_ids))
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, rope=self.rope, position_ids=position_ids, attention_mask=attention_mask,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    """Backbone + LM head tied to the token embedding."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def forward(self, input_ids, targets=None, attention_mask=None, position_ids=None, kv_caches=None):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      position_ids=position_ids, kv_caches=kv_caches)
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss, new_caches


def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, init_std) everywhere; residual-branch output projections scaled by 1/sqrt(2*n_layer)."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            if hasattr(blk.mixer, "o_proj"):  blk.mixer.o_proj.weight.mul_(scale)
            if hasattr(blk.cmix, "down_proj"): blk.cmix.down_proj.weight.mul_(scale)
            if hasattr(blk.cmix, "fc2"):       blk.cmix.fc2.weight.mul_(scale)

In [12]:
# Cell 11 — Utilities: parameter count, optimizer groups (FIXED), save / load
def count_params(model: nn.Module) -> Dict[str, int]:
    """Tied embedding counted once (model.parameters() de-duplicates shared tensors)."""
    total = sum(p.numel() for p in model.parameters())
    emb = model.backbone.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Weight decay on 2-D weight matrices only (attention/MLP projections).
    No decay on norm gains/biases (1-D) and on the embedding.
    (Model 01 matched '.b' in names, which also matched 'backbone.blocks...' -> decay was never applied.)"""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model, cfg, extra=None):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, tmp)
    os.replace(tmp, path)


def load_model(path, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

### Model tests
Parameter counts of both presets, then behaviour tests on a **tiny** copy of the architecture (window 8 so that the
sliding window really triggers on 32-token inputs). The tests that would have caught model 01's missing RoPE are
#7 and #8.

In [13]:
# Cell 12 — Model tests
def run_model_tests():
    # ---- A. parameter budget (real presets) --------------------------------------------------------------
    pb = count_params(LegalLM(preset_baseline_rope()))
    ph = count_params(LegalLM(preset_hybrid_gqa()))
    print(f"[A] baseline  total={pb['total']/1e6:.3f}M | hybrid_gqa total={ph['total']/1e6:.3f}M "
          f"(difference {ph['total']-pb['total']:+,d} params)")
    assert 50_000_000 <= ph["total"] <= 300_000_000, "outside the 50M-300M rule"
    assert abs(ph["total"] - pb["total"]) <= 0.001 * pb["total"], "presets are not parameter-matched"

    # ---- B. behaviour tests on a tiny copy -----------------------------------------------------------------
    cfg = ModelConfig(vocab_size=300, n_layer=4, d_model=128, n_head=4, n_kv_head=2, head_dim=32, d_ff=256,
                      max_seq_len=128, local_window=8, global_layers=[3], init_std=0.08)
    torch.manual_seed(0)
    model = LegalLM(cfg).to(device); init_weights(model, cfg); model.eval()
    B, T = 2, 32
    ids = torch.randint(0, cfg.vocab_size, (B, T), device=device)
    with torch.no_grad():
        full, loss, _ = model(ids, targets=ids)
    print(f"[1] init loss {loss.item():.3f} (ln V = {math.log(cfg.vocab_size):.3f})")

    # 2. causality
    with torch.no_grad():
        ids2 = ids.clone(); ids2[:, -1] = (ids2[:, -1] + 1) % cfg.vocab_size
        l2, _, _ = model(ids2)
    d = (full[:, :-1] - l2[:, :-1]).abs().max().item()
    print(f"[2] causality: max |dlogits| on earlier positions = {d:.2e}"); assert d < 1e-5

    # 3. the sliding window really limits the receptive field (single layers)
    rope = model.backbone.rope
    pos = torch.arange(T, device=device).unsqueeze(0)
    x = torch.randn(1, T, cfg.d_model, device=device)
    x2 = x.clone(); x2[:, 0] += 1.0                       # perturb position 0 only
    with torch.no_grad():
        for li, expect_far_change in [(0, False), (3, True)]:   # layer 0 = local, layer 3 = global
            att = model.backbone.blocks[li].mixer
            y1, _ = att(x, rope=rope, position_ids=pos)
            y2, _ = att(x2, rope=rope, position_ids=pos)
            far = (y1[:, 8:] - y2[:, 8:]).abs().max().item()       # positions >= window cannot see position 0
            near = (y1[:, :8] - y2[:, :8]).abs().max().item()
            print(f"[3] layer {li} ({'global' if expect_far_change else 'local '}): change at pos<8 {near:.2e} | pos>=8 {far:.2e}")
            assert near > 1e-6
            assert (far > 1e-6) == expect_far_change, "window / global behaviour wrong"

    # 4. KV cache == full forward (prefill 1 token, then 1 token at a time)
    with torch.no_grad():
        lp, _, caches = model(ids[:, :1])
        outs = [lp]
        for t in range(1, T):
            lg, _, caches = model(ids[:, t:t + 1], position_ids=torch.full((B, 1), t, device=device), kv_caches=caches)
            outs.append(lg)
        d = (torch.cat(outs, 1) - full).abs().max().item()
        print(f"[4a] KV cache (token-by-token) max |dlogits| = {d:.2e}"); assert d < 1e-4
        lp, _, caches = model(ids[:, :20])                  # prefill 20 (> window), then continue
        outs = [lp]
        for t in range(20, T):
            lg, _, caches = model(ids[:, t:t + 1], kv_caches=caches)       # positions inferred from the cache
            outs.append(lg)
        d = (torch.cat(outs, 1) - full).abs().max().item()
        print(f"[4b] KV cache (prefill 20 + steps) max |dlogits| = {d:.2e}"); assert d < 1e-4

    # 5. SDPA == manual attention
    cfg_m = replace(cfg, use_sdpa=False)
    mm = LegalLM(cfg_m).to(device); mm.load_state_dict(model.state_dict()); mm.eval()
    with torch.no_grad():
        d = (mm(ids)[0] - full).abs().max().item()
    print(f"[5] SDPA vs manual max |dlogits| = {d:.2e}"); assert d < 1e-4

    # 6. left padding does not change real-token logits
    Lp = 6
    padded = torch.cat([torch.zeros(1, Lp, dtype=torch.long, device=device), ids[0:1]], 1)
    am = torch.cat([torch.zeros(1, Lp, dtype=torch.long, device=device), torch.ones(1, T, dtype=torch.long, device=device)], 1)
    pp = torch.cat([torch.zeros(1, Lp, dtype=torch.long, device=device), torch.arange(T, device=device).unsqueeze(0)], 1)
    with torch.no_grad():
        lg, _, _ = model(padded, attention_mask=am, position_ids=pp)
    d = (lg[0, Lp:] - full[0]).abs().max().item()
    print(f"[6] left-padding equivalence max |dlogits| = {d:.2e}"); assert d < 1e-4 and not torch.isnan(lg).any()

    # 7. RoPE is really used (same weights, pos_type='none' must give different logits)
    cfg_n = replace(cfg, pos_type="none")
    mn = LegalLM(cfg_n).to(device); mn.load_state_dict(model.state_dict()); mn.eval()
    with torch.no_grad():
        d = (mn(ids)[0] - full).abs().max().item()
    print(f"[7] RoPE vs no-position max |dlogits| = {d:.2e}  (must be clearly > 0)"); assert d > 1e-3

    # 8. RoPE is relative: shifting all positions by a constant leaves logits unchanged
    with torch.no_grad():
        shifted, _, _ = model(ids, position_ids=(torch.arange(T, device=device) + 37).unsqueeze(0).expand(B, T))
    d = (shifted - full).abs().max().item()
    print(f"[8] position-shift invariance max |dlogits| = {d:.2e}"); assert d < 1e-3

    # 9. GQA shapes + checkpoint round trip
    a0 = model.backbone.blocks[0].mixer
    assert a0.k_proj.out_features == cfg.n_kv_head * cfg.head_dim and a0.q_proj.out_features == cfg.n_head * cfg.head_dim
    tmp = WORK_DIR / "_test_ckpt.pt"
    save_model(tmp, model, cfg)
    m2, c2, _ = load_model(tmp, map_location=device); m2 = m2.to(device).eval()
    with torch.no_grad():
        d = (m2(ids)[0] - full).abs().max().item()
    print(f"[9] GQA shapes ok; checkpoint round trip max |dlogits| = {d:.2e}"); assert d < 1e-6
    tmp.unlink()
    print("\nAll model tests PASSED")

run_model_tests()

[A] baseline  total=54.932M | hybrid_gqa total=54.932M (difference +0 params)
[1] init loss 4.501 (ln V = 5.704)
[2] causality: max |dlogits| on earlier positions = 0.00e+00
[3] layer 0 (local ): change at pos<8 8.13e-01 | pos>=8 0.00e+00
[3] layer 3 (global): change at pos<8 6.80e-01 | pos>=8 2.62e-01
[4a] KV cache (token-by-token) max |dlogits| = 2.82e-06
[4b] KV cache (prefill 20 + steps) max |dlogits| = 2.61e-06
[5] SDPA vs manual max |dlogits| = 2.15e-06
[6] left-padding equivalence max |dlogits| = 1.91e-06
[7] RoPE vs no-position max |dlogits| = 2.54e+00  (must be clearly > 0)
[8] position-shift invariance max |dlogits| = 2.86e-06
[9] GQA shapes ok; checkpoint round trip max |dlogits| = 0.00e+00

All model tests PASSED


In [14]:
# Cell 13 — KV-cache budget: what GQA + local attention buy (numbers for the report)
def kv_bytes_per_seq(cfg: ModelConfig, seq=1024, bytes_per=2, rolling_local=False):
    total_tokens = 0
    for i in range(cfg.n_layer):
        w = cfg.window_for(i)
        total_tokens += min(seq, w) if (rolling_local and w) else seq
    return 2 * cfg.n_kv_head * cfg.head_dim * bytes_per * total_tokens

b, h = preset_baseline_rope(), preset_hybrid_gqa()
print(f"model 01 style (MHA, full)        : {kv_bytes_per_seq(b)/1e6:6.2f} MB / sequence (fp16, 1024 tokens)")
print(f"hybrid_gqa, full caches (this nb) : {kv_bytes_per_seq(h)/1e6:6.2f} MB  -> {kv_bytes_per_seq(b)/kv_bytes_per_seq(h):.1f}x smaller (GQA only)")
print(f"hybrid_gqa, rolling local caches  : {kv_bytes_per_seq(h, rolling_local=True)/1e6:6.2f} MB  -> "
      f"{kv_bytes_per_seq(b)/kv_bytes_per_seq(h, rolling_local=True):.1f}x smaller (theoretical, NOT implemented here)")

model 01 style (MHA, full)        :  25.17 MB / sequence (fp16, 1024 tokens)
hybrid_gqa, full caches (this nb) :   6.29 MB  -> 4.0x smaller (GQA only)
hybrid_gqa, rolling local caches  :   2.75 MB  -> 9.1x smaller (theoretical, NOT implemented here)


### Data
Same data recipe as model 01 (same four sources, same token caps, same document-hash validation split), so the
shards are byte-identical to the ones you already built. The cell below **reuses shards if it finds them**
(in `/kaggle/working/shards`, or in an attached Kaggle dataset) and only re-encodes from Hugging Face if it
does not. Re-encoding takes a while (~150M tokens through a pure-Python BPE).

**Tip:** after the first encode, save `/kaggle/working/shards` as a Kaggle Dataset (≈290 MB) and attach it to every
later run, so you never re-encode.

In [15]:
# Cell 14 — Find the shards, or build them
SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]

SOURCE_PATHS = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}
TOKEN_CAPS = {"acts": 20_000_000, "high_courts": 65_000_000, "supreme_court": 25_000_000, "tribunals": 40_000_000}
SHARD_TOKENS, VAL_FRACTION, VAL_CAP_TOTAL = 50_000_000, 0.005, 5_000_000
HF_REPO = "Exploration-Lab/CS-781-Capstone"


def find_shards() -> Optional[Path]:
    cands = [WORK_DIR / "shards"]
    for pat in ["kaggle/input/*", "kaggle/input/*/*", "kaggle/input/*/*/*"]:
        for d in sorted(Path("/").glob(pat)):
            cands += [d, d / "shards"]
    for d in cands:
        if all((d / s / "meta.json").exists() for s in SOURCES):
            return d
    return None


def clean_text(t): return (t or "").replace("\r\n", "\n").strip()

def is_val(doc_id, salt="phase1"):
    h = int(hashlib.md5((salt + "::" + doc_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000.0 < VAL_FRACTION


def encode_all(shard_dir: Path):
    """Identical to model 01's encoder: stream each JSON from HF, BPE-encode, write uint16 shards + val.bin."""
    import ijson
    from array import array
    from huggingface_hub import HfFileSystem
    from tqdm.auto import tqdm

    def stream_docs(path_in_repo):
        fs = HfFileSystem()
        with fs.open(f"datasets/{HF_REPO}/{path_in_repo}", "rb") as f:
            for i, doc in enumerate(ijson.items(f, "item")):
                if isinstance(doc, dict):
                    doc.setdefault("_idx", i)
                    yield doc

    for source, path_in_repo in SOURCE_PATHS.items():
        out_dir = shard_dir / source
        if (out_dir / "meta.json").exists():
            print(f"[{source}] already done, skipping"); continue
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n[{source}] streaming {path_in_repo}")
        meta = {"source": source, "hf_path": path_in_repo, "shards": [], "total_tokens": 0, "total_bytes": 0, "num_docs": 0}
        shard_ids, val_ids, shard_idx = array("H"), array("H"), 0
        val_cap, cap = VAL_CAP_TOTAL // len(SOURCE_PATHS), TOKEN_CAPS[source]

        def flush():
            nonlocal shard_ids, shard_idx
            if len(shard_ids) == 0: return
            arr = np.frombuffer(shard_ids, dtype=np.uint16)
            final = out_dir / f"shard_{shard_idx:04d}.bin"
            arr.tofile(final.with_suffix(".bin.tmp")); final.with_suffix(".bin.tmp").rename(final)
            meta["shards"].append({"name": final.name, "tokens": int(arr.size)})
            meta["total_tokens"] += int(arr.size)
            print(f"  [{source}] {final.name}: {arr.size/1e6:.1f}M tokens")
            shard_ids, shard_idx = array("H"), shard_idx + 1

        pbar = tqdm(desc=f"  {source}", unit="doc")
        try:
            for doc in stream_docs(path_in_repo):
                pbar.update(1)
                text = clean_text(doc.get("text"))
                if not text: continue
                doc_id = str(doc.get("url") or doc.get("id") or doc.get("_idx"))
                meta["num_docs"] += 1
                meta["total_bytes"] += len(text.encode("utf-8"))
                ids = tokenizer.encode(text, allow_special=False)
                if is_val(doc_id) and len(val_ids) < val_cap:
                    val_ids.append(BOS_ID); val_ids.extend(ids); val_ids.append(EOS_ID)
                    continue
                shard_ids.append(BOS_ID); shard_ids.extend(ids); shard_ids.append(EOS_ID)
                if len(shard_ids) >= SHARD_TOKENS: flush()
                if meta["total_tokens"] + len(shard_ids) >= cap:
                    print(f"  [{source}] hit cap {cap/1e6:.0f}M"); break
        finally:
            pbar.close()
        flush()
        if len(val_ids) > 0:
            (out_dir / "val.bin").write_bytes(np.frombuffer(val_ids, dtype=np.uint16).tobytes())
            meta["val_tokens"] = int(len(val_ids))
        (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
        print(f"[{source}] {meta['total_tokens']/1e6:.1f}M tokens, {meta['num_docs']} docs, val={meta.get('val_tokens', 0)/1e6:.2f}M")


SHARD_DIR = find_shards()
if SHARD_DIR is None:
    SHARD_DIR = WORK_DIR / "shards"
    print("No shards found -> encoding from Hugging Face into", SHARD_DIR)
    encode_all(SHARD_DIR)
    print("\nDone. TIP: save /kaggle/working/shards as a Kaggle Dataset so later runs can skip this step.")
else:
    print("Using existing shards at", SHARD_DIR)

for s in SOURCES:
    m = json.loads((SHARD_DIR / s / "meta.json").read_text())
    print(f"{s:14s} train={m['total_tokens']/1e6:7.2f}M  val={m.get('val_tokens', 0)/1e6:5.2f}M  docs={m['num_docs']:>7,}")

No shards found -> encoding from Hugging Face into /kaggle/working/shards

[acts] streaming Phase_1_Pretraining/Train/train_acts.json


  acts: 0doc [00:00, ?doc/s]

  [acts] hit cap 20M
  [acts] shard_0000.bin: 20.0M tokens
[acts] 20.0M tokens, 2228 docs, val=0.07M

[high_courts] streaming Phase_1_Pretraining/Train/train_high_courts.json


  high_courts: 0doc [00:00, ?doc/s]

  [high_courts] shard_0000.bin: 50.0M tokens
  [high_courts] hit cap 65M
  [high_courts] shard_0001.bin: 15.0M tokens
[high_courts] 65.0M tokens, 112740 docs, val=0.31M

[supreme_court] streaming Phase_1_Pretraining/Train/train_supreme_court.json


  supreme_court: 0doc [00:00, ?doc/s]

  [supreme_court] hit cap 25M
  [supreme_court] shard_0000.bin: 25.0M tokens
[supreme_court] 25.0M tokens, 3650 docs, val=0.12M

[tribunals] streaming Phase_1_Pretraining/Train/train_tribunals.json


  tribunals: 0doc [00:00, ?doc/s]

  [tribunals] hit cap 40M
  [tribunals] shard_0000.bin: 40.0M tokens
[tribunals] 40.0M tokens, 8466 docs, val=0.20M

Done. TIP: save /kaggle/working/shards as a Kaggle Dataset so later runs can skip this step.
acts           train=  20.01M  val= 0.07M  docs=  2,228
high_courts    train=  65.00M  val= 0.31M  docs=112,740
supreme_court  train=  25.00M  val= 0.12M  docs=  3,650
tribunals      train=  40.00M  val= 0.20M  docs=  8,466


In [16]:
# Cell 15 — Data loaders
@dataclass
class MixConfig:
    source_weights: Dict[str, float]

    @classmethod
    def from_sizes(cls, sizes: Dict[str, int]):
        """Sampling probability proportional to the number of tokens per source (same as model 01)."""
        keys = list(sizes)
        arr = np.array([sizes[k] for k in keys], dtype=np.float64)
        return cls(dict(zip(keys, (arr / arr.sum()).tolist())))


class ShardReader:
    """Memory-mapped shards (+ validation tokens) of one source."""
    def __init__(self, source_dir: Path):
        self.dir = Path(source_dir)
        meta = json.loads((self.dir / "meta.json").read_text())
        self.shards = meta["shards"]
        self.total_tokens = sum(s["tokens"] for s in self.shards)
        self.arrays = [np.memmap(self.dir / s["name"], dtype=np.uint16, mode="r") for s in self.shards]
        w = np.array([s["tokens"] for s in self.shards], dtype=np.float64)
        self.shard_weights = w / w.sum()
        vp = self.dir / "val.bin"
        self.val_arr = np.memmap(vp, dtype=np.uint16, mode="r") if vp.exists() else None


class MixedWindowLoader:
    """Random (source, shard, offset) windows of seq_len+1 tokens. A pure function of (seed, step): resume-safe.
    Batch size = micro_batch, taken from TrainConfig (model 01 had it hard-wired to 8 in the loader)."""
    def __init__(self, sources, mix: MixConfig, seq_len: int, micro_batch: int, seed: int = 0):
        self.sources, self.seq_len, self.micro_batch, self.seed = sources, seq_len, micro_batch, seed
        self.keys = list(sources)
        self.p = np.array([mix.source_weights[k] for k in self.keys])

    def get_batch(self, step: int):
        rng = np.random.default_rng((self.seed * 1_000_003 + step * 2_654_435_761) & 0xFFFFFFFF)
        B, T = self.micro_batch, self.seq_len
        x = np.empty((B, T), dtype=np.int64); y = np.empty((B, T), dtype=np.int64)
        for b in range(B):
            r = self.sources[rng.choice(self.keys, p=self.p)]
            arr = r.arrays[rng.choice(len(r.arrays), p=r.shard_weights)]
            start = int(rng.integers(0, max(1, len(arr) - (T + 1))))
            w = np.asarray(arr[start:start + T + 1], dtype=np.int64)
            if w.size < T + 1:
                w = np.concatenate([w, np.zeros(T + 1 - w.size, dtype=np.int64)])
            x[b], y[b] = w[:-1], w[1:]
        return torch.from_numpy(x), torch.from_numpy(y)


class FixedValLoader:
    """Non-overlapping validation windows, batched PER SOURCE (so per-source losses are exact).
    Same windows as model 01's val loader -> the overall val loss is comparable with 01's."""
    def __init__(self, sources, seq_len: int, windows_per_source: int, batch_size: int = 8):
        self.batch_size, self.items = batch_size, []
        for name, r in sources.items():
            if r.val_arr is None: continue
            n = min(windows_per_source, len(r.val_arr) // (seq_len + 1))
            ws = [np.asarray(r.val_arr[i * (seq_len + 1):(i + 1) * (seq_len + 1)], dtype=np.int64) for i in range(n)]
            if ws: self.items.append((name, ws))

    def __iter__(self):
        for name, ws in self.items:
            for i in range(0, len(ws), self.batch_size):
                a = np.stack(ws[i:i + self.batch_size])
                yield name, torch.from_numpy(a[:, :-1].copy()), torch.from_numpy(a[:, 1:].copy())


def build_readers(shard_root: Path):
    return {s: ShardReader(shard_root / s) for s in SOURCES if (shard_root / s / "meta.json").exists()}

readers = build_readers(SHARD_DIR)
mix = MixConfig.from_sizes({k: v.total_tokens for k, v in readers.items()})
print("mix:", {k: round(v, 3) for k, v in mix.source_weights.items()})

mix: {'acts': 0.133, 'high_courts': 0.433, 'supreme_court': 0.167, 'tribunals': 0.267}


In [17]:
# Cell 16 — Evidence for the window size: how long are paragraphs / sentences in YOUR data (in tokens)?
def paragraph_stats(per_source_docs=150):
    for s, r in readers.items():
        if r.val_arr is None: continue
        arr = np.asarray(r.val_arr[:3_000_000])
        starts = np.where(arr == BOS_ID)[0]
        lens = []
        for a, b in list(zip(starts[:-1], starts[1:]))[:per_source_docs]:
            doc = tokenizer.decode([int(t) for t in arr[a + 1:b - 1]])
            for para in doc.split("\n"):
                if len(para.split()) >= 5:
                    lens.append(len(tokenizer.encode(para, allow_special=False)))
        if lens:
            q = np.percentile(lens, [50, 75, 90, 95])
            print(f"{s:14s} paragraphs={len(lens):6d} | tokens per paragraph  median={q[0]:.0f}  p75={q[1]:.0f}  p90={q[2]:.0f}  p95={q[3]:.0f}")
    print("\nCompare with the window: tokens inside the window are always visible; anything farther back is only")
    print("reachable through the 3 global layers (and through stacking local layers).")

paragraph_stats()

acts           paragraphs=  1348 | tokens per paragraph  median=29  p75=52  p90=82  p95=106
high_courts    paragraphs=  1326 | tokens per paragraph  median=46  p75=71  p90=124  p95=174
supreme_court  paragraphs=   761 | tokens per paragraph  median=106  p75=193  p90=306  p95=372
tribunals      paragraphs=  2394 | tokens per paragraph  median=52  p75=97  p90=170  p95=235

Compare with the window: tokens inside the window are always visible; anything farther back is only
reachable through the 3 global layers (and through stacking local layers).


### Training
Differences from the notebook for model 01 (all deliberate):
* fp16 autocast + `GradScaler` really enabled on the GPU
* weight decay really applied (the param-group printout shows how many params are decayed: should be ~38M)
* batch size comes from `TrainConfig` (micro_batch × grad_accum sequences of 1024 tokens)
* validation loss is exact per source; an evaluation always runs at the very end; `best.pt` = lowest val loss
* `last.pt` is written atomically every few minutes so a crashed session can be resumed

Optimiser settings (AdamW β=(0.9,0.95), peak LR 6e-4 → 6e-5 cosine, 400 warm-up steps, clip 1.0, wd 0.1) and
**32,768 tokens per optimiser step** are identical to what model 01 *actually* used.

In [18]:
# Cell 17 — Training code
@dataclass
class TrainConfig:
    seq_len: int = 1024
    micro_batch: int = 16
    grad_accum: int = 2                # micro_batch * grad_accum = 32 sequences = 32,768 tokens per step (same as 01)
    peak_lr: float = 6e-4
    min_lr: float = 6e-5
    warmup_steps: int = 400
    # warmup_steps: int = 10
    total_steps: int = 6000
    # total_steps: int = 40
    weight_decay: float = 0.1
    betas: Tuple[float, float] = (0.9, 0.95)
    eps: float = 1e-8
    grad_clip: float = 1.0
    eval_every: int = 250
    # eval_every: int = 10
    val_windows_per_source: int = 64
    log_every: int = 50
    # log_every: int = 5
    ckpt_every_minutes: float = 20
    # ckpt_every_minutes: float = 2
    time_budget_hours: float = 9.0    # hard stop (Kaggle sessions end at 12h)
    # time_budget_hours: float = 0.1    # hard stop (Kaggle sessions end at 12h)
    seed: int = 1337
    output_dir: str = "runs/arch02_hybrid_gqa"

    @property
    def tokens_per_step(self): return self.micro_batch * self.grad_accum * self.seq_len


def cosine_lr(step, warmup, total, peak, minimum):
    if step < warmup:
        return peak * (step + 1) / warmup
    if step >= total:
        return minimum
    prog = (step - warmup) / max(1, total - warmup)
    return minimum + 0.5 * (peak - minimum) * (1 + math.cos(math.pi * prog))


@torch.no_grad()
def evaluate(model, val_loader):
    """Returns (overall loss, {source: loss}). Overall = token-weighted mean over all val windows."""
    model.eval()
    acc = {}
    for src, x, y in val_loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_FP16):
            _, loss, _ = model(x, targets=y)
        a = acc.setdefault(src, [0.0, 0])
        a[0] += loss.item() * y.numel(); a[1] += y.numel()
    model.train()
    per_src = {s: a[0] / a[1] for s, a in acc.items()}
    overall = sum(a[0] for a in acc.values()) / sum(a[1] for a in acc.values())
    return overall, per_src


def _atomic_save(obj, path: Path):
    tmp = path.with_suffix(".tmp"); torch.save(obj, tmp); os.replace(tmp, path)


def train(cfg: ModelConfig, tcfg: TrainConfig, readers, mix, val_loader, resume_from: Optional[Path] = None):
    out_dir = Path(tcfg.output_dir); out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "log.jsonl"
    (out_dir / "config.json").write_text(json.dumps({"model": asdict(cfg), "train": asdict(tcfg)}, indent=1))

    train_loader = MixedWindowLoader(readers, mix, tcfg.seq_len, tcfg.micro_batch, tcfg.seed)
    torch.manual_seed(tcfg.seed)
    model = LegalLM(cfg).to(device)
    init_weights(model, cfg)
    pc = count_params(model)
    groups = build_param_groups(model, tcfg.weight_decay)
    n_dec, n_nodec = (sum(p.numel() for p in g["params"]) for g in groups)
    print(f"params: total {pc['total']/1e6:.2f}M | weight-decayed {n_dec/1e6:.2f}M | not decayed {n_nodec/1e6:.2f}M")
    assert n_dec > 0.5 * pc["total"], "weight decay is not being applied to the matrices!"
    assert 50_000_000 <= pc["total"] <= 300_000_000, "outside the 50M-300M rule"

    opt = torch.optim.AdamW(groups, lr=tcfg.peak_lr, betas=tcfg.betas, eps=tcfg.eps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)

    start_step = 0
    if resume_from is not None:
        ck = torch.load(resume_from, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start_step = ck["step"] + 1
        print(f"resumed from {resume_from} at step {start_step}")

    model.train()
    t0, last_ckpt_t, tokens_seen, best_val = time.time(), time.time(), 0, float("inf")
    if (out_dir / "best.pt").exists() and resume_from is not None:
        best_val = torch.load(out_dir / "best.pt", map_location="cpu")["extra"].get("val_loss", best_val)

    def do_eval(step):
        nonlocal best_val
        val, per = evaluate(model, val_loader)
        print(f"  [eval] step {step} val_loss={val:.4f} ppl={math.exp(val):.2f} per_src={ {k: round(v, 3) for k, v in per.items()} }")
        with open(log_path, "a") as f:
            f.write(json.dumps({"step": step, "val_loss": val, "val_per_src": per}) + "\n")
        if val < best_val:
            best_val = val
            save_model(out_dir / "best.pt", model, cfg, extra={"step": step, "val_loss": val})

    step = start_step - 1
    for step in range(start_step, tcfg.total_steps):
        lr = cosine_lr(step, tcfg.warmup_steps, tcfg.total_steps, tcfg.peak_lr, tcfg.min_lr)
        for g in opt.param_groups: g["lr"] = lr
        opt.zero_grad(set_to_none=True)
        step_loss = 0.0
        for micro in range(tcfg.grad_accum):
            x, y = train_loader.get_batch(step * tcfg.grad_accum + micro)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_FP16):
                _, loss, _ = model(x, targets=y)
            scaler.scale(loss / tcfg.grad_accum).backward()
            step_loss += loss.item() / tcfg.grad_accum
            tokens_seen += y.numel()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
        scaler.step(opt); scaler.update()

        if step % tcfg.log_every == 0:
            dt = time.time() - t0
            mem = f" | {torch.cuda.max_memory_allocated()/1e9:.1f}GB" if device.type == "cuda" else ""
            print(f"step {step:5d}/{tcfg.total_steps} | loss {step_loss:.4f} | lr {lr:.2e} | gnorm {gn:.2f} "
                  f"| {tokens_seen/dt/1000:.1f}k tok/s | {dt/60:.1f}m{mem}")
            with open(log_path, "a") as f:
                f.write(json.dumps({"step": step, "loss": step_loss, "lr": lr, "grad_norm": float(gn),
                                    "tokens_seen": tokens_seen, "elapsed_s": dt}) + "\n")

        if (step + 1) % tcfg.eval_every == 0 and step + 1 < tcfg.total_steps:
            do_eval(step)

        if (time.time() - last_ckpt_t) / 60 >= tcfg.ckpt_every_minutes:
            _atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                          "step": step, "config": asdict(cfg), "train_config": asdict(tcfg)}, out_dir / "last.pt")
            last_ckpt_t = time.time()
            print(f"  [ckpt] last.pt saved at step {step}")

        if (time.time() - t0) / 3600 >= tcfg.time_budget_hours:
            print(f"time budget reached at step {step}"); break

    do_eval(step)                                           # always evaluate at the very end
    save_model(out_dir / "final.pt", model, cfg, extra={"step": step})
    print(f"\nfinished at step {step}. best val loss {best_val:.4f} (ppl {math.exp(best_val):.2f}). files in {out_dir}")
    return model, out_dir

### Run settings, speed benchmark, training
* `MATCHED_RUN=True` trains exactly 6000 steps = the ≈197M tokens model 01 really saw (its loader silently used 8
  sequences per micro-batch, so 6000 × 32,768 tokens): the cleanest comparison for the report.
* `MATCHED_RUN=False` fills `TRAIN_HOURS` of GPU time with as many steps as fit (better model, not matched to 01).
* The benchmark measures real tokens/second on your GPU and picks the largest micro-batch that fits in memory
  (16 → 8 → 4, keeping 32 sequences per optimiser step), then computes `TOTAL_STEPS`.

In [19]:
# Cell 18 — RUN SETTINGS  (the only cell you may want to edit)
ARCH = "hybrid_gqa"       # "hybrid_gqa" = architecture 02  |  "baseline_rope" = controlled baseline (MHA, full attention, RoPE on)
MATCHED_RUN = False       # True -> exactly 6000 steps (same token budget as model 01);  False -> fill TRAIN_HOURS
TRAIN_HOURS = 7.0         # used only when MATCHED_RUN is False (leave a few hours of the 12h session for generation)
SEQS_PER_STEP = 32        # sequences of 1024 tokens per optimiser step (= 32,768 tokens, same as model 01)
RESUME_FROM = None        # e.g. WORK_DIR / "runs" / "arch02_hybrid_gqa" / "last.pt" to continue a crashed run

cfg = preset_hybrid_gqa() if ARCH == "hybrid_gqa" else preset_baseline_rope()
RUN_NAME = {"hybrid_gqa": "arch02_hybrid_gqa", "baseline_rope": "baseline_rope"}[ARCH]
print("ARCH:", ARCH, "| run dir:", WORK_DIR / "runs" / RUN_NAME)

ARCH: hybrid_gqa | run dir: /kaggle/working/runs/arch02_hybrid_gqa


In [20]:
# Cell 19 — Speed benchmark -> micro-batch + TOTAL_STEPS
def benchmark(cfg, micro_batch, grad_accum, seq_len=1024, n_steps=6, warmup=2):
    loader = MixedWindowLoader(readers, mix, seq_len, micro_batch, seed=0)
    model = LegalLM(cfg).to(device); init_weights(model, cfg); model.train()
    opt = torch.optim.AdamW(build_param_groups(model, 0.1), lr=1e-4, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    t0 = None
    for s in range(warmup + n_steps):
        if s == warmup:
            if device.type == "cuda": torch.cuda.synchronize()
            t0 = time.time()
        opt.zero_grad(set_to_none=True)
        for m in range(grad_accum):
            x, y = loader.get_batch(s * grad_accum + m)
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_FP16):
                _, loss, _ = model(x, targets=y)
            scaler.scale(loss / grad_accum).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
    if device.type == "cuda": torch.cuda.synchronize()
    dt = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
    del model, opt
    if device.type == "cuda": torch.cuda.empty_cache()
    return n_steps * grad_accum * micro_batch * seq_len / dt, peak


import gc
MICRO, tok_s = None, None
for mb in (16, 8, 4):
    gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    try:
        tok_s, peak = benchmark(cfg, mb, SEQS_PER_STEP // mb)
        MICRO = mb
        print(f"micro_batch={mb:2d} x grad_accum={SEQS_PER_STEP // mb:2d}: {tok_s/1000:.1f}k tok/s, peak GPU memory {peak:.1f} GB")
        break
    except torch.cuda.OutOfMemoryError:
        print(f"micro_batch={mb}: out of memory, trying smaller")
        torch.cuda.empty_cache()
assert MICRO is not None, "even micro_batch=4 does not fit"
ACCUM = SEQS_PER_STEP // MICRO
TOK_PER_STEP = SEQS_PER_STEP * 1024

if MATCHED_RUN:
    TOTAL_STEPS = 6000
else:
    TOTAL_STEPS = int(TRAIN_HOURS * 3600 * 0.93 * tok_s / TOK_PER_STEP) // 250 * 250   # 7% reserved for evals/checkpoints
    TOTAL_STEPS = max(TOTAL_STEPS, 1000)
#  remove this while full run :)
# TOTAL_STEPS = 40
train_tokens = TOTAL_STEPS * TOK_PER_STEP
corpus_tokens = sum(r.total_tokens for r in readers.values())
print(f"\nTOTAL_STEPS = {TOTAL_STEPS}  ->  {train_tokens/1e6:.0f}M training tokens = {train_tokens/corpus_tokens:.2f} epochs over the "
      f"{corpus_tokens/1e6:.0f}M-token corpus  |  estimated time {train_tokens/tok_s/3600/0.93:.1f} h")
if train_tokens / corpus_tokens > 4:
    print("WARNING: more than ~4 epochs over the same data; the model may start to overfit (watch the val loss). "
          "Consider a smaller TRAIN_HOURS, or more data (raise TOKEN_CAPS and re-encode).")

micro_batch=16 x grad_accum= 2: 18.0k tok/s, peak GPU memory 13.6 GB

TOTAL_STEPS = 12750  ->  418M training tokens = 2.79 epochs over the 150M-token corpus  |  estimated time 6.9 h


In [21]:
# Cell 20 — TRAIN  (checkpoints: best.pt, final.pt, last.pt, log.jsonl in the run dir)
tcfg = TrainConfig(micro_batch=MICRO, grad_accum=ACCUM, total_steps=TOTAL_STEPS,
                   output_dir=str(WORK_DIR / "runs" / RUN_NAME))
val_loader = FixedValLoader(readers, tcfg.seq_len, tcfg.val_windows_per_source)
print(f"tokens per optimiser step: {tcfg.tokens_per_step:,} | model 01 final val loss for reference: 2.1168 (step 5750)")
model, run_dir = train(cfg, tcfg, readers, mix, val_loader, resume_from=RESUME_FROM)

tokens per optimiser step: 32,768 | model 01 final val loss for reference: 2.1168 (step 5750)
params: total 54.93M | weight-decayed 38.54M | not decayed 16.40M
step     0/12750 | loss 10.4271 | lr 1.50e-06 | gnorm 16.20 | 17.2k tok/s | 0.0m | 13.6GB
step    50/12750 | loss 7.6459 | lr 7.65e-05 | gnorm 1.99 | 17.6k tok/s | 1.6m | 13.6GB
step   100/12750 | loss 5.3885 | lr 1.51e-04 | gnorm 2.34 | 17.7k tok/s | 3.1m | 13.6GB
step   150/12750 | loss 4.2179 | lr 2.26e-04 | gnorm 1.55 | 17.7k tok/s | 4.7m | 13.7GB
step   200/12750 | loss 4.2120 | lr 3.01e-04 | gnorm 1.40 | 17.7k tok/s | 6.2m | 13.7GB
  [eval] step 249 val_loss=3.9494 ppl=51.91 per_src={'acts': 4.461, 'high_courts': 2.694, 'supreme_court': 4.622, 'tribunals': 4.02}
step   250/12750 | loss 3.7333 | lr 3.76e-04 | gnorm 1.32 | 17.5k tok/s | 7.8m | 13.7GB
step   300/12750 | loss 3.4654 | lr 4.51e-04 | gnorm 1.43 | 17.6k tok/s | 9.4m | 13.7GB
step   350/12750 | loss 3.2690 | lr 5.26e-04 | gnorm 1.15 | 17.6k tok/s | 10.9m | 13.7GB


In [22]:
# Cell 21 — Training curves (for the report)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def plot_training(run_dir):
    tr, va = [], []
    for line in open(Path(run_dir) / "log.jsonl"):
        r = json.loads(line)
        if "loss" in r: tr.append((r["step"], r["loss"]))
        if "val_loss" in r: va.append((r["step"], r["val_loss"]))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(*zip(*tr), label="train loss (per 50 steps)", alpha=0.6)
    if va: ax.plot(*zip(*va), "o-", label="val loss")
    ax.set_xlabel("step"); ax.set_ylabel("cross-entropy"); ax.set_ylim(1.5, 5); ax.legend(); ax.grid(alpha=0.3)
    out = Path(run_dir) / "curves.png"; fig.savefig(out, dpi=130, bbox_inches="tight"); plt.close(fig)
    print("saved", out)

plot_training(run_dir)

saved /kaggle/working/runs/arch02_hybrid_gqa/curves.png


### Generation and submissions
Fixes relative to the generation code of model 01:
* `<pad>/<bos>/<eos>/<unk>` can never be sampled → every output reaches 128 words (01 produced outputs with as few as 1 word)
* `no_repeat_n`: a generated n-gram can never occur twice (attacks the repetition loops of greedy decoding)
* repetition penalty is vectorised (01's version did a GPU sync per token)
* prompts get the same `\r\n → \n` + strip normalisation as the training text
* every submission is checked: all IDs present, no extra IDs, word counts, zip structure

In [23]:
# Cell 22 — generate()
@torch.no_grad()
def generate(model, prompts: List[List[int]], max_new_tokens=300, temperature=0.0, top_k=0, top_p=1.0,
             repetition_penalty=1.0, no_repeat_n=0) -> List[List[int]]:
    """Batched generation with KV cache. Left-pads the prompts. temperature == 0 -> greedy.
    Returns exactly max_new_tokens token ids per prompt (specials are banned, so no early stop)."""
    model.eval()
    B, L = len(prompts), max(len(p) for p in prompts)
    input_ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=device)
    attn_mask = torch.zeros((B, L), dtype=torch.long, device=device)
    pos_ids = torch.zeros((B, L), dtype=torch.long, device=device)
    for i, p in enumerate(prompts):
        input_ids[i, L - len(p):] = torch.tensor(p, device=device)
        attn_mask[i, L - len(p):] = 1
        pos_ids[i, L - len(p):] = torch.arange(len(p), device=device)

    logits, _, caches = model(input_ids, attention_mask=attn_mask, position_ids=pos_ids)
    next_logits = logits[:, -1, :]
    cur_pos = pos_ids[:, -1:] + 1
    gen = torch.empty((B, 0), dtype=torch.long, device=device)        # generated ids (for the repetition penalty)
    gen_lists = [[] for _ in range(B)]                                 # same, as python lists (for n-gram blocking)
    seen = [dict() for _ in range(B)]                                  # n-1 gram prefix -> set of next tokens already used
    NEG = -float("inf")

    for step in range(max_new_tokens):
        lg = next_logits.float()
        for t in (PAD_ID, BOS_ID, EOS_ID, UNK_ID):                     # never generate special tokens
            lg[:, t] = NEG
        if repetition_penalty != 1.0 and gen.shape[1] > 0:             # CTRL-style penalty on tokens generated so far
            s = lg.gather(1, gen)
            lg.scatter_(1, gen, torch.where(s > 0, s / repetition_penalty, s * repetition_penalty))
        if no_repeat_n > 1:                                            # block any repeated n-gram (within the generation)
            for i in range(B):
                g = gen_lists[i]
                if len(g) >= no_repeat_n - 1:
                    banned = seen[i].get(tuple(g[len(g) - (no_repeat_n - 1):]))
                    if banned:
                        lg[i, list(banned)] = NEG

        if temperature == 0.0:
            nxt = lg.argmax(dim=-1)
        else:
            lg = lg / temperature
            if top_k > 0:
                v, _ = torch.topk(lg, top_k, dim=-1)
                lg[lg < v[:, [-1]]] = NEG
            if top_p < 1.0:
                srt, idx = torch.sort(lg, descending=True, dim=-1)
                cdf = srt.softmax(-1).cumsum(-1)
                drop = cdf > top_p
                drop[:, 1:] = drop[:, :-1].clone(); drop[:, 0] = False  # always keep the best token
                srt[drop] = NEG
                lg = srt.scatter(-1, idx, srt)
            nxt = torch.multinomial(lg.softmax(-1), 1).squeeze(-1)

        gen = torch.cat([gen, nxt.view(B, 1)], dim=1)
        for i, t in enumerate(nxt.tolist()):
            g = gen_lists[i]; g.append(t)
            if no_repeat_n > 1 and len(g) >= no_repeat_n:
                seen[i].setdefault(tuple(g[-no_repeat_n:-1]), set()).add(t)

        if step == max_new_tokens - 1:
            break
        attn_mask = torch.cat([attn_mask, torch.ones(B, 1, device=device, dtype=torch.long)], dim=1)
        logits, _, caches = model(nxt.view(B, 1), attention_mask=attn_mask, position_ids=cur_pos, kv_caches=caches)
        next_logits = logits[:, -1, :]
        cur_pos = cur_pos + 1
    return gen_lists

In [24]:
# Cell 23 — Test prompts, prediction builder, submission writer with checks
from datasets import load_dataset

def load_test_prompts_hf() -> Dict[str, str]:
    """The Phase-1 test split is ONE row whose columns are the prompt IDs '1'...'1000', each {"text": ...}."""
    ds = load_dataset(HF_REPO, "phase_1", split="test", streaming=True, token=True)
    out = {}
    for row in ds:
        for k, v in row.items():
            if isinstance(v, dict): out[str(k)] = v.get("text", "")
            elif isinstance(v, str): out[str(k)] = v
    return out


def first_words(text: str, n: int = 128) -> str:
    return " ".join(text.split()[:n])


@torch.no_grad()
def make_predictions(model, prompts: Dict[str, str], max_new_tokens=300, batch_size=16, **gen_kw) -> Dict[str, dict]:
    model.eval()
    ids = list(prompts)
    max_prompt = model.cfg.max_seq_len - max_new_tokens
    enc = []
    for k in ids:
        text = prompts[k].replace("\r\n", "\n").strip()                # same normalisation as the training text
        enc.append(tokenizer.encode(text, allow_special=False)[-max_prompt:])   # keep the most recent context
    order = sorted(range(len(ids)), key=lambda i: -len(enc[i]))        # long -> short: little padding per batch
    preds, t0 = {}, time.time()
    for bi in range(0, len(order), batch_size):
        batch = order[bi:bi + batch_size]
        outs = generate(model, [enc[j] for j in batch], max_new_tokens=max_new_tokens, **gen_kw)
        for j, o in zip(batch, outs):
            preds[ids[j]] = {"text": first_words(tokenizer.decode(o), 128)}     # 128 WORDS, continuation only
        if (bi // batch_size) % 10 == 0:
            print(f"  {min(bi + batch_size, len(order))}/{len(order)} prompts | {time.time()-t0:.0f}s")
    return preds


def distinct_ngram(text, n=4):
    w = text.split(); g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return len(set(g)) / max(1, len(g))


def write_submission(preds: Dict[str, dict], prompts: Dict[str, str], out_dir: Path):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    assert set(preds) == set(prompts), "ID mismatch: missing or extra IDs!"
    (out_dir / "prediction.json").write_text(json.dumps(preds, ensure_ascii=False))
    zp = out_dir / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()       # file must sit at the zip root
    wc = [len(v["text"].split()) for v in preds.values()]
    d4 = float(np.mean([distinct_ngram(v["text"]) for v in preds.values()]))
    print(f"  wrote {zp} | {len(preds)} ids | words min/mean/max = {min(wc)}/{sum(wc)/len(wc):.1f}/{max(wc)} "
          f"| outputs <128 words: {sum(w < 128 for w in wc)} | distinct-4gram ratio {d4:.3f} (1.0 = no repetition)")

In [25]:
# Cell 24 — Load the best checkpoint and the 1000 test prompts
best_path = WORK_DIR / "runs" / RUN_NAME / "best.pt"
gen_model, gen_cfg, extra = load_model(best_path, map_location=device)
gen_model = gen_model.to(device).eval()               # fp32 for generation (safest)
print(f"loaded {best_path.name}: step {extra.get('step')}, val_loss {extra.get('val_loss')}")
prompts = load_test_prompts_hf()
print(len(prompts), "prompts | sample id:", next(iter(prompts)))

loaded best.pt: step 12749, val_loss 1.9441224113106728


README.md:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

1000 prompts | sample id: 1


In [26]:
# Cell 25 — Three decoding variants of the SAME model = three distinct submissions
# Each takes roughly 10-25 minutes on a T4. Try v1 first; v2/v3 are decoding experiments for the report.
VARIANTS = {
    "v1_greedy_rp1.1_nr6":  dict(temperature=0.0, repetition_penalty=1.1, no_repeat_n=6),
    "v2_T0.7_p0.9_rp1.1_nr6": dict(temperature=0.7, top_p=0.9, repetition_penalty=1.1, no_repeat_n=6),
    "v3_T0.5_p0.9_nr4":     dict(temperature=0.5, top_p=0.9, repetition_penalty=1.0, no_repeat_n=4),
}
for name, kw in VARIANTS.items():
    print(f"\n=== {name}  {kw}")
    torch.manual_seed(1234)
    preds = make_predictions(gen_model, prompts, max_new_tokens=300, batch_size=16, **kw)
    write_submission(preds, prompts, WORK_DIR / "submissions" / name)
print("\nUpload /kaggle/working/submissions/<variant>/submission.zip to Codabench (one file per submission).")


=== v1_greedy_rp1.1_nr6  {'temperature': 0.0, 'repetition_penalty': 1.1, 'no_repeat_n': 6}
  16/1000 prompts | 6s
  176/1000 prompts | 71s
  336/1000 prompts | 132s
  496/1000 prompts | 192s
  656/1000 prompts | 252s
  816/1000 prompts | 311s
  976/1000 prompts | 370s
  wrote /kaggle/working/submissions/v1_greedy_rp1.1_nr6/submission.zip | 1000 ids | words min/mean/max = 49/127.7/128 | outputs <128 words: 9 | distinct-4gram ratio 0.929 (1.0 = no repetition)

=== v2_T0.7_p0.9_rp1.1_nr6  {'temperature': 0.7, 'top_p': 0.9, 'repetition_penalty': 1.1, 'no_repeat_n': 6}
  16/1000 prompts | 7s
  176/1000 prompts | 73s
  336/1000 prompts | 137s
  496/1000 prompts | 200s
  656/1000 prompts | 262s
  816/1000 prompts | 324s
  976/1000 prompts | 385s
  wrote /kaggle/working/submissions/v2_T0.7_p0.9_rp1.1_nr6/submission.zip | 1000 ids | words min/mean/max = 72/127.8/128 | outputs <128 words: 7 | distinct-4gram ratio 0.987 (1.0 = no repetition)

=== v3_T0.5_p0.9_nr4  {'temperature': 0.5, 'top_p': 0

In [27]:
# Cell 26 — Peek at a few outputs (sanity check by eye)
_p = json.loads((WORK_DIR / "submissions" / "v1_greedy_rp1.1_nr6" / "prediction.json").read_text())
for k in list(_p)[:3]:
    print(f"[{k}] prompt ends: ...{prompts[k][-250:]!r}")
    print(f"     continuation: {_p[k]['text'][:400]!r}\n")

[19] prompt ends: ...'ppeals. 12.Mr. U.R. Lalit, learned senior counsel appearing for the appellants submitted before us that the evidence against the 35 accused persons being similar in nature, the Courts below committed an error of law and facts in acquitting the 21 out'
     continuation: 'of the 35 accused persons. He submitted that the High Court has committed an error of law and facts which are not in dispute. 13. Having heard learned counsel for the parties, we are of the opinion that the High Court has rightly held that the prosecution has been able to prove its case beyond reasonable doubt against the accused persons. The High Court has committed a grave error of law and fact '

[80] prompt ends: ...'y in Ramsden v. Dyson (supra) can prevail against the requirement of formalities laid down in the Victorian Statute referred to above any more than the equity in Maddison v. Alderson (supra)can do against the requirements of the [Transfer of Property'
     continuation: 'Act] (4) 

### Optional: local evaluation with the official script
Builds held-out prompts/references from the validation documents (first 512 words = prompt, next 128 words =
reference) and generates with one decoding variant. Score the two files with the **official evaluation script**
(Drive link in the capstone PDF, §18.1). Both files use the Codabench format `{"id": {"text": ...}}`.

In [28]:
# Cell 27 — Build a local held-out set and generate for it
def build_local_eval(per_src=25):
    prompts_l, refs = {}, {}
    for s, r in readers.items():
        if r.val_arr is None: continue
        arr = np.asarray(r.val_arr[:])
        starts = np.where(arr == BOS_ID)[0]
        n = 0
        for a, b in zip(starts, list(starts[1:]) + [len(arr)]):
            if n >= per_src: break
            words = tokenizer.decode([int(t) for t in arr[a + 1:b] if t not in (BOS_ID, EOS_ID)]).split()
            if len(words) >= 640:
                prompts_l[f"{s}_{n}"] = " ".join(words[:512])
                refs[f"{s}_{n}"] = {"text": " ".join(words[512:640])}
                n += 1
    return prompts_l, refs

lp, lr_ = build_local_eval()
print(len(lp), "held-out prompts")
out_l = WORK_DIR / "local_eval"; out_l.mkdir(exist_ok=True)
(out_l / "ref.json").write_text(json.dumps(lr_, ensure_ascii=False))
lpred = make_predictions(gen_model, lp, max_new_tokens=300, batch_size=16, **VARIANTS["v1_greedy_rp1.1_nr6"])
(out_l / "pred_v1.json").write_text(json.dumps(lpred, ensure_ascii=False))
print("wrote", out_l / "ref.json", "and", out_l / "pred_v1.json", "-> score them with the official evaluation script")

77 held-out prompts
  16/77 prompts | 6s
wrote /kaggle/working/local_eval/ref.json and /kaggle/working/local_eval/pred_v1.json -> score them with the official evaluation script
